# 🏢 Week 14 hands-on: Multi-tenant governance

## Learning Objectives
In this notebook, you will learn:
1. **One platform, three teams** - run three teams' evaluations through one API that decides access, records every decision and charges each team
2. **Roles and tenant isolation** - call the API as every principal, and probe another team's data with guessed references and IDs
3. **An immutable registry** - version datasets by content hash and refuse edits, through the API and around it
4. **Redaction before storage** - mask personal data at the write and count what reached the database file
5. **Audit and attribution** - verify a hash-chained audit log, tamper with it, and read per-team costs

## Where this fits
The theory is in `../theory/week-14-multi-tenant-governance-theory.md`, and the lab walkthrough with exercises and solutions in `week-14-multi-tenant-governance-handson.md`. This notebook runs the same code as `lab.py`, one step per cell. The platform itself is `shared/governance.py`.

## Prerequisites
- The course `.venv` as the kernel (see `setup.md`). No API key, no cost.
- Weeks 4 and 11: the retrieval benchmark and the evaluation runner, loaded by path and run unchanged.
- Weeks 12 and 13: failing closed on missing evidence, and identity-aware redaction.


---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/`, imports the platform from `shared/governance.py` and Week 13's redaction patterns, and loads the week's tenant policy and evaluator definitions.


In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Platform, redaction patterns, tenant policy and evaluators
# ============================================================================
import hashlib
import importlib.util
import json
import re
import sqlite3
import sys
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

from shared.governance import (  # noqa: E402
    GENESIS, PUBLIC_DENIAL, Denied, IntegrityError, Platform, canonical, redact_records,
)
from shared.support_agent import load_world  # noqa: E402
from shared.tracing import CARD_NUMBER, CUSTOMER_ID, EMAIL  # noqa: E402

HERE = COURSE / "weeks" / "week-14-multi-tenant-governance" / "handson"
DATA = HERE.parent / "dataset"
RUNS = HERE / "runs" / "week-14"


TENANTS = json.loads((DATA / "tenants-v1.json").read_text())


EVALUATORS = json.loads((DATA / "evaluators-v1.json").read_text())


NAMES = tuple(c["name"] for c in load_world()["customers"].values())


RETRIEVAL_COST_PER_QUERY = 0.0002


print(f"tenants: {list(TENANTS['tenants'])}")
print(f"principals: {len(TENANTS['principals'])}, roles: {list(TENANTS['roles'])}")
print("✅ Environment ready")


---
## 🏢 Part 1: One Platform, Three Teams

Each team brings a runner from an earlier week. `run_retrieval` calls Week 4's benchmark, and `run_tools` and `run_faults` call Week 11's runner, loaded by path and unchanged. Each returns metrics and a SIMULATED cost. The fault runner defaults to the lax backend: trust and safety runs it as a safety probe, as Week 12 did.


In [ ]:
# ============================================================================
# RUNNERS: Week 4 and Week 11, loaded by path
# ============================================================================
def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


def load_week_lab(folder: str, name: str):
    """Earlier weeks' labs run unchanged, loaded by path (Week 11's pattern)."""
    spec = importlib.util.spec_from_file_location(name, COURSE / "weeks" / folder / "handson" / "lab.py")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


W4 = load_week_lab("week-04-retrieval-evaluation", "week04_lab_for_w14")


W11 = load_week_lab("week-11-evaluation-harness-and-ci", "week11_lab_for_w14")


def run_retrieval(records: list[dict], retriever: str = "hybrid", k: int = 5) -> dict:
    rows = W4.run_benchmark(retriever, k, queries=records)
    return {"metrics": {"cases": len(rows), "recall_at_5": round(W4.mean(rows, "recall"), 4),
                        "ndcg_at_5": round(W4.mean(rows, "ndcg"), 4),
                        "stale_or_restricted_hits": sum(r["violations"] for r in rows)},
            "cost_usd": len(rows) * RETRIEVAL_COST_PER_QUERY}


def suite_metrics(artifact: dict) -> dict:
    summary = artifact["summary"]
    return {"cases": artifact["completed_cases"],
            "pass_rate": round(summary["status"]["pass"] / artifact["completed_cases"], 4),
            **summary["safety"]}


def run_tools(records: list[dict], agent: str = "v2", strict: bool = True) -> dict:
    artifact = W11.run_suite("tools", agent, strict, cases=records)
    return {"metrics": suite_metrics(artifact), "cost_usd": artifact["summary"]["cost_usd"]}


def run_faults(records: list[dict], agent: str = "stable_key", strict: bool = False) -> dict:
    """Trust and safety runs the fault suite as a safety probe: on the lax backend (Week 12)."""
    artifact = W11.run_suite("faults", agent, strict, cases=records)
    return {"metrics": suite_metrics(artifact), "cost_usd": artifact["summary"]["cost_usd"]}


RUNNERS = {"retrieval": run_retrieval, "tools": run_tools, "faults": run_faults}


print(run_retrieval(read_jsonl(DATA / "escalation-queries-v1.jsonl")))


`build_platform` gives `Platform` the tenant policy, a SIMULATED clock, the runners and the customer names to mask. `seed` registers each team's datasets and evaluators, and trust and safety shares `refund-safety` with the refunds team. `SCRIPT` is the working day.


In [ ]:
# ============================================================================
# THE DAY: Register, share, run, and one run refused on budget
# ============================================================================
class SimClock:
    """SIMULATED wall clock: every reading is one minute after the last, from 2026-03-16 09:00 UTC."""

    def __init__(self, start: str = "2026-03-16T09:00:00+00:00"):
        self.t = datetime.fromisoformat(start)

    def __call__(self) -> str:
        now, self.t = self.t, self.t + timedelta(minutes=1)
        return now.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def build_platform(db: str = ":memory:") -> Platform:
    return Platform(TENANTS, db=db, clock=SimClock(), runners=RUNNERS, names=NAMES)


def seed(platform: Platform) -> dict:
    """Each team registers its datasets and evaluators; trust-safety shares refund-safety."""
    refs = {
        "policy-queries": platform.register_dataset(
            "rag.engineer", "policy-queries", read_jsonl(COURSE / "datasets/retrieval/policy-queries-v1.jsonl")),
        "escalation-queries": platform.register_dataset(
            "rag.engineer", "escalation-queries", read_jsonl(DATA / "escalation-queries-v1.jsonl")),
        "tool-cases": platform.register_dataset(
            "refunds.engineer", "tool-cases", read_jsonl(COURSE / "datasets/tools/tool-cases-v1.jsonl")),
        "fault-cases": platform.register_dataset(
            "safety.engineer", "fault-cases", read_jsonl(COURSE / "datasets/faults/fault-cases-v1.jsonl")),
    }
    for entry in EVALUATORS["evaluators"]:
        refs[entry["name"]] = platform.register_evaluator(entry["owner"], entry["name"], entry["spec"])
        for grantee in entry["share_with"]:
            platform.share_evaluator(entry["owner"], refs[entry["name"]], grantee)
    return refs


SCRIPT = [  # (actor, dataset, evaluators, runner arguments): one SIMULATED working day
    ("rag.engineer", "policy-queries", ["retrieval-quality"], {}),
    ("rag.engineer", "escalation-queries", ["retrieval-quality"], {}),
    ("refunds.engineer", "tool-cases", ["task-success", "refund-safety"], {"agent": "v1"}),
    ("refunds.engineer", "tool-cases", ["task-success", "refund-safety"], {"agent": "v2"}),
    ("safety.engineer", "fault-cases", ["refund-safety"], {"agent": "naive_retry"}),
    ("safety.engineer", "fault-cases", ["refund-safety"], {"agent": "stable_key"}),
    ("refunds.engineer", "tool-cases", ["task-success", "refund-safety"], {"agent": "v2", "strict": False}),
]


def run_script(platform: Platform, refs: dict) -> list[dict]:
    """Run SCRIPT. A denied run is recorded as {"denied": message} so the day's story stays complete."""
    out = []
    for actor, dataset, evaluators, args in SCRIPT:
        try:
            out.append(platform.run_eval(actor, refs[dataset], [refs[e] for e in evaluators], **args))
        except Denied as exc:
            out.append({"actor": actor, "dataset": refs[dataset], "runner_args": args, "denied": str(exc)})
    return out


def build_day(db: str = ":memory:") -> tuple[Platform, dict, list[dict]]:
    platform = build_platform(db)
    refs = seed(platform)
    return platform, refs, run_script(platform, refs)


def print_demo(refs: dict, runs: list[dict]) -> None:
    print("Registered (SIMULATED platform, fictional teams):")
    for ref in refs.values():
        print(f"  {ref}")
    print(f"{'run':9} {'actor':17} {'dataset':34} {'args':30} {'verdicts':38} cost $")
    for r in runs:
        args = json.dumps(r["runner_args"], separators=(",", ":"))
        if "denied" in r:
            print(f"{'-':9} {r['actor']:17} {r['dataset']:34} {args:30} DENIED: {r['denied']}")
            continue
        verdicts = ", ".join(f"{e['ref'].split('/')[1].split('@')[0]} {e['verdict']}" for e in r["evaluators"])
        print(f"{r['run_id']:9} {r['actor']:17} {r['dataset']['ref']:34} {args:30} {verdicts:38} {r['cost_usd']:.4f}")


platform, refs, runs = build_day()
print_demo(refs, runs)


Six runs completed and one was refused before it started. Run-0003 fails `refund-safety` on v1's wrong-order refunds and false success claims, and run-0005 fails it on naive-retry duplicates. The refunds team names trust and safety's evaluator by its full ref, `trust-safety/refund-safety@v1`. Its third run, a safety probe, reserved more than the budget had left.


---
## 🔐 Part 2: Roles and Tenant Isolation

`access_matrix` calls nine operations as each of the nine principals. Every cell below is a real call through `_authorize`. It runs on a fresh copy of the day, and so does the probe, so the day's own audit log stays as it was for Part 5.


In [ ]:
# ============================================================================
# ACCESS MATRIX: Every principal against the same calls
# ============================================================================
def access_matrix(platform: Platform, refs: dict, runs: list[dict]) -> dict:
    """Call the API as every principal. Own-tenant targets come from the caller's tenant."""
    first_run = {r["tenant"]: r["run_id"] for r in reversed(runs) if "run_id" in r}
    own_dataset = {"support-rag": refs["policy-queries"], "refunds-agent": refs["tool-cases"],
                   "trust-safety": refs["fault-cases"]}
    own_evaluator = {"support-rag": "retrieval-quality", "refunds-agent": "task-success",
                     "trust-safety": "refund-safety"}
    other = {"support-rag": "refunds-agent", "refunds-agent": "support-rag", "trust-safety": "support-rag"}
    spec = {e["name"]: e["spec"] for e in EVALUATORS["evaluators"]}
    matrix = {}
    for actor, entry in TENANTS["principals"].items():
        t = entry["tenant"] if entry["tenant"] != "*" else "support-rag"
        calls = {
            "read own dataset": lambda: platform.read_dataset(actor, own_dataset[t]),
            "write dataset": lambda: platform.register_dataset(actor, "scratch", [{"note": "access check"}]),
            "write evaluator": lambda: platform.register_evaluator(actor, own_evaluator[t], spec[own_evaluator[t]]),
            "read own run": lambda: platform.read_run(actor, first_run[t]),
            "read other team's dataset": lambda: platform.read_dataset(actor, own_dataset[other[t]]),
            "read other team's run": lambda: platform.read_run(actor, first_run[other[t]]),
            "use refund-safety": lambda: platform.read_evaluator(actor, refs["refund-safety"]),
            "read usage": lambda: platform.usage(actor),
            "read audit": lambda: platform.audit(actor),
        }
        row = {}
        for label, call in calls.items():
            try:
                call()
                row[label] = "allow"
            except Denied:
                row[label] = "deny"
        matrix[actor] = row
    return matrix


def print_matrix(matrix: dict) -> None:
    labels = list(next(iter(matrix.values())))
    print(f"{'principal':17} " + " ".join(f"{i + 1:>3}" for i in range(len(labels))))
    for actor, row in matrix.items():
        print(f"{actor:17} " + " ".join(f"{'yes' if row[l] == 'allow' else ' - ':>3}" for l in labels))
    for i, label in enumerate(labels, 1):
        print(f"  {i}. {label}")


matrix_platform, matrix_refs, matrix_runs = build_day()
print_matrix(access_matrix(matrix_platform, matrix_refs, matrix_runs))


Nobody reads another team's dataset or run (columns 5 and 6). The grant makes `refund-safety` usable by the refunds team (column 7). The auditor reads usage and the log, and no data. Next, the refunds engineer probes other teams with guessed references and run IDs.


In [ ]:
# ============================================================================
# PROBES: Guessed references and IDs, and what the owner sees
# ============================================================================
PROBES = [  # what refunds.engineer tries against support-rag and trust-safety
    ("read_dataset", "support-rag/escalation-queries@v1"),
    ("read_dataset", "support-rag/escalation-queries"),
    ("read_dataset", "support-rag/no-such-dataset@v1"),
    ("read_dataset", "trust-safety/fault-cases@v1"),
    ("read_run", "run-0001"),
    ("read_run", "run-0099"),
    ("read_evaluator", "support-rag/retrieval-quality@v1"),
    ("register_evaluator", "refund-safety"),
]


def probe(platform: Platform, actor: str = "refunds.engineer") -> list[tuple[str, str, str]]:
    """Each probe's public answer. Every one is denied, and missing and forbidden read the same."""
    out = []
    for method, target in PROBES:
        try:
            if method == "register_evaluator":
                platform.register_evaluator(actor, target, {"rules": []})
            else:
                getattr(platform, method)(actor, target)
            out.append((method, target, "ALLOWED"))
        except Denied as exc:
            out.append((method, target, str(exc)))
    return out


probe_platform, _probe_refs, _probe_runs = build_day()
for method, target, answer in probe(probe_platform):
    print(f"refunds.engineer {method:18} {target:36} -> {answer}")
seen = [e for e in probe_platform.audit("rag.owner") if e["decision"] == "deny"]
print(f"rag.owner's audit view: {len(seen)} denied events on support-rag resources")
for e in seen:
    print(f"  seq {e['seq']:>3} {e['actor']} {e['action']} {e['resource']} reason={e['reason']}")


Seven probes get the same answer whether the target exists or not. The log keeps the true reason, `cross_tenant` or `missing`, and the support team's owner sees the four probes on its resources.


---
## 🗂️ Part 3: An Immutable Registry

`registry_story` registers the refunds team's cases three times, then tries two edits that skip the API: an `UPDATE` through SQL, and the same edit after dropping the trigger that refused it. It runs on its own platform because it drops a trigger.


In [ ]:
# ============================================================================
# REGISTRY: Versions by content hash, and edits refused
# ============================================================================
def registry_story(platform: Platform) -> list[str]:
    """Register, re-register, change, try to edit in place, then tamper behind the API's back."""
    lines = []
    cases = read_jsonl(COURSE / "datasets/tools/tool-cases-v1.jsonl")
    lines.append(f"register tool-cases                  -> {platform.register_dataset('refunds.engineer', 'tool-cases', cases)}")
    lines.append(f"register the same 30 cases again     -> {platform.register_dataset('refunds.engineer', 'tool-cases', cases)}")
    changed = [dict(c, risk="critical") if c["case_id"] == "t-023" else c for c in cases]
    lines.append(f"register with t-023 risk changed     -> {platform.register_dataset('refunds.engineer', 'tool-cases', changed)}")
    v1 = platform.read_dataset("refunds.engineer", "tool-cases@v1")
    lines.append(f"read tool-cases@v1, t-023 risk       -> {next(c['risk'] for c in v1 if c['case_id'] == 't-023')}")
    try:
        platform.db.execute("UPDATE artifacts SET body = '[]' WHERE name = 'tool-cases' AND version = 1")
    except sqlite3.DatabaseError as exc:
        lines.append(f"UPDATE tool-cases v1 in the database -> refused: {exc}")
    platform.db.execute("DROP TRIGGER artifacts_no_update")          # someone with database access
    platform.db.execute("UPDATE artifacts SET body = ? WHERE name = 'tool-cases' AND version = 1",
                        (json.dumps(cases[:29]),))
    try:
        platform.read_dataset("refunds.engineer", "tool-cases@v1")
    except IntegrityError as exc:
        lines.append(f"read after a direct edit             -> refused: {exc}")
    return lines


story_platform, _story_refs, _story_runs = build_day()
print("\n".join(registry_story(story_platform)))


Identical content keeps `@v1`, changed content gets `@v2`, and `@v1` still reads the old value. The trigger stops the first edit, and the hash check on read catches the second.


---
## 🙈 Part 4: Redaction Before Storage

`redaction_report` counts personal data in the raw escalation queries, after pattern-only redaction, and in the bytes of a SQLite file holding a whole day: datasets, run records and the audit log.


In [ ]:
# ============================================================================
# REDACTION: What reaches the database file
# ============================================================================
def count_personal_data(text: str) -> dict:
    return {"emails": len(EMAIL.findall(text)), "card numbers": len(CARD_NUMBER.findall(text)),
            "customer IDs": len(CUSTOMER_ID.findall(text)), "names": sum(text.count(n) for n in NAMES)}


def redaction_report(db_path: Path) -> dict:
    """Count personal data in the raw upload, in a pattern-only redaction, and in the database file's bytes."""
    raw = read_jsonl(DATA / "escalation-queries-v1.jsonl")
    db_path.parent.mkdir(parents=True, exist_ok=True)
    if db_path.exists():
        db_path.unlink()
    platform, _refs, _runs = build_day(str(db_path))
    platform.db.close()
    stored = db_path.read_bytes().decode("utf-8", errors="ignore")
    return {"raw upload": count_personal_data(json.dumps(raw)),
            "patterns only": count_personal_data(json.dumps(redact_records(raw))),
            "database file": count_personal_data(stored),
            "customer_id fields": len(re.findall(r'"customer_id": "cust-\d+"', stored)),
            "recall raw": run_retrieval(raw)["metrics"]["recall_at_5"],
            "recall redacted": run_retrieval(redact_records(raw, NAMES))["metrics"]["recall_at_5"]}


report = redaction_report(RUNS / "platform.db")
for where in ("raw upload", "patterns only", "database file"):
    print(f"{where:16} {report[where]}")
print(f"customer_id fields in the file: {report['customer_id fields']}")
print(f"escalation-queries recall@5: raw {report['recall raw']}, as stored {report['recall redacted']}")


Patterns remove emails, card numbers and the typed customer ID and leave all 6 names. The platform also masks the names it knows, and the file holds none. The 41 customer IDs are the synthetic `customer_id` fields the runners replay. On these eight queries, redaction left recall at 5 unchanged.


---
## 🧾 Part 5: Audit and Attribution

The anchor is the head of the log at the end of the day, kept outside the database. The auditor's read is itself an event. Deleting the budget denial at seq 46 then breaks the chain.


In [ ]:
# ============================================================================
# AUDIT: Verify, anchor, delete one event
# ============================================================================
def audit_summary(events: list[dict]) -> Counter:
    return Counter((e["decision"], e["reason"]) for e in events)


def delete_event(platform: Platform, seq: int) -> None:
    """Tamper 1: remove one event, as someone with direct database access."""
    platform.db.execute("DROP TRIGGER IF EXISTS audit_no_delete")
    platform.db.execute("DELETE FROM audit WHERE seq = ?", (seq,))


def rewrite_chain(platform: Platform) -> None:
    """Tamper 2: renumber the remaining events and recompute every hash, so the chain verifies again."""
    platform.db.execute("DROP TRIGGER IF EXISTS audit_no_update")
    rows = platform.db.execute("SELECT seq, event FROM audit ORDER BY seq").fetchall()
    platform.db.execute("DELETE FROM audit")
    prev = GENESIS
    for new_seq, (_seq, event) in enumerate(rows, start=1):
        digest = hashlib.sha256(prev.encode() + canonical(json.loads(event))).hexdigest()
        platform.db.execute("INSERT INTO audit VALUES (?, ?, ?, ?)", (new_seq, event, prev, digest))
        prev = digest


anchor = platform.audit_head()
events = platform.audit("platform.auditor")
print(f"anchor: seq {anchor[0]}, hash {anchor[1][:16]}...; the auditor reads {len(events)} events")
for (decision, reason), n in sorted(audit_summary(events).items()):
    print(f"  {decision:6} {reason or '-':18} {n}")
print(f"verify: {platform.verify_audit(anchor) or 'chain intact'}")
delete_event(platform, 46)
print(f"after deleting seq 46: {platform.verify_audit()}")


In [ ]:
# ============================================================================
# USAGE: Cost charged to the team that ran each evaluation
# ============================================================================
for u in platform.usage("platform.auditor"):
    print(u)


Trust and safety paid for its own two runs only, although the refunds team ran its evaluator twice. The refunds team's denial is in the log with the arithmetic behind it.


---
## 🧪 Your Own Cases

Edit the cell below. It registers a shadow `refund-safety` with no rules in the refunds team's namespace and runs v1 with the short and the full reference (exercise 3). Try the other exercises here too: share with `support-rag`, or change a reserve in `TENANTS` and rerun `build_day()`.


In [ ]:
# ============================================================================
# YOUR OWN CASES: A shadow evaluator in the caller's namespace
# ============================================================================
own = build_platform()
own_refs = seed(own)
print(own.register_evaluator("refunds.owner", "refund-safety", {"rules": []}))
for ref in ["refund-safety", "trust-safety/refund-safety"]:
    run = own.run_eval("refunds.engineer", own_refs["tool-cases"], [ref], agent="v1")
    print(ref, [(e["ref"], e["verdict"], e["sha256"][:12]) for e in run["evaluators"]])


---
## 🐞 Drill 1: The API That Trusted the Caller's Tenant

`read_run_trusting_caller` checks the role and compares the run's tenant with a tenant the caller supplies. It also reads the database directly, so nothing is logged.


In [ ]:
# ============================================================================
# DRILL 1: Break, symptom, fix
# ============================================================================
def read_run_trusting_caller(platform: Platform, actor: str, tenant: str, run_id: str) -> dict:
    """DRILL 1, broken on purpose: the caller says which tenant it acts for, and the code believes it."""
    row = platform.db.execute("SELECT tenant, body FROM runs WHERE run_id=?", (run_id,)).fetchone()
    p = platform.principal(actor)
    if p is None or "run.read" not in TENANTS["roles"][p.role] or row is None or row[0] != tenant:
        raise Denied(PUBLIC_DENIAL)
    return json.loads(row[1])


before = len(platform.audit("platform.auditor"))
leaked = read_run_trusting_caller(platform, "refunds.engineer", "support-rag", "run-0001")
after = len(platform.audit("platform.auditor"))
print(f"broken: refunds.engineer read {leaked['run_id']} of {leaked['tenant']}; new events besides the two audit reads: {after - before - 1}")
try:
    platform.read_run("refunds.engineer", "run-0001")
except Denied as exc:
    print(f"fixed: {exc}; logged as {platform.audit('platform.auditor')[-2]['reason']}")


---
## 🐞 Drill 2: The Audit Log That Verified After Tampering

Part 5 deleted seq 46 and the chain caught it. Here the tamperer also renumbers the later events and recomputes every hash.


In [ ]:
# ============================================================================
# DRILL 2: Rewrite the chain, then check the anchor
# ============================================================================
rewrite_chain(platform)
print(f"without the anchor: {platform.verify_audit() or 'chain intact'}")
print(f"with the anchor:    {platform.verify_audit(anchor)}")


A rewritten chain is consistent with itself. Only the head stored outside the database shows that seq 46 changed.


---
## 📝 Summary

**One platform, three teams.** Every method takes only the caller's identity, decides in `_authorize`, logs the decision and charges the caller's team.

**Roles and isolation.** Nobody reads another team's data, grants open one evaluator for use and not for change, and missing and forbidden look the same to the caller.

**The registry.** Content hashes decide versions, triggers stop edits through the platform, and a hash check on read catches edits around it. Short names resolve in the caller's namespace, so gates must use full refs and hashes.

**Redaction.** Mask at the write, including known names, and prove it by scanning the stored bytes.

**Audit and attribution.** The chain catches edits and gaps, the anchor catches truncation and rewrites, and costs go to the team that ran the evaluation.

**Next Steps.** Week 15 asks how far to trust these numbers: `week-15-reliable-measurement-and-statistics-handson.ipynb` runs repeated stochastic evaluations and paired bootstrap comparisons.
